# Proyek Pengembangan Machine Learning Pipeline

Notebook ini membangun pipeline machine learning sederhana menggunakan TensorFlow Extended (TFX). Dataset yang digunakan adalah Iris Dataset untuk klasifikasi tiga spesies bunga Iris berdasarkan ukuran sepal dan petal.


## 1. Persiapan Library dan Path

Pipeline dijalankan menggunakan `InteractiveContext`. Seluruh artifact pipeline disimpan pada folder `fahrual_19-pipeline`, sesuai kriteria submission.


In [11]:
import os

import tensorflow as tf
import tensorflow_model_analysis as tfma
from tfx.components import (
    CsvExampleGen,
    Evaluator,
    ExampleValidator,
    Pusher,
    SchemaGen,
    StatisticsGen,
    Trainer,
    Transform,
)
from tfx.dsl.components.common.resolver import Resolver
from tfx.dsl.experimental import latest_blessed_model_resolver
from tfx.orchestration.experimental.interactive.interactive_context import InteractiveContext
from tfx.proto import pusher_pb2, trainer_pb2
from tfx.types import Channel
from tfx.types.standard_artifacts import Model, ModelBlessing

PIPELINE_NAME = "fahrual_19"
PIPELINE_ROOT = os.path.abspath("fahrual_19-pipeline")
DATA_ROOT = os.path.abspath("data")
TRANSFORM_MODULE = os.path.abspath(os.path.join("modules", "transform.py"))
TRAINER_MODULE = os.path.abspath(os.path.join("modules", "trainer.py"))
SERVING_MODEL_DIR = os.path.abspath("serving_model")

print("TensorFlow version:", tf.__version__)
print("Pipeline root:", PIPELINE_ROOT)


TensorFlow version: 2.13.1
Pipeline root: D:\SERTIFIKAT DICODING\Mlops\Proyek pengembangan machine learning pipeline\fahrual_19-pipeline


## 2. Membuat InteractiveContext

`InteractiveContext` digunakan agar setiap komponen TFX dapat dijalankan dan diperiksa langsung dari notebook.


In [12]:
context = InteractiveContext(pipeline_root=PIPELINE_ROOT)


## 3. ExampleGen

Komponen `CsvExampleGen` membaca data CSV dari folder `data` dan mengubahnya menjadi format TFRecord yang digunakan oleh komponen berikutnya.


In [13]:
example_gen = CsvExampleGen(input_base=DATA_ROOT)
context.run(example_gen)


ExecutionResult(
    component_id: CsvExampleGen
    execution_id: 11
    outputs:
        examples: OutputChannel(artifact_type=Examples, producer_component_id=CsvExampleGen, output_key=examples, additional_properties={}, additional_custom_properties={}, _input_trigger=None)

## 4. StatisticsGen, SchemaGen, dan ExampleValidator

Tahap ini menghitung statistik data, membuat schema otomatis, dan memeriksa anomali pada dataset.


In [14]:
statistics_gen = StatisticsGen(examples=example_gen.outputs["examples"])
context.run(statistics_gen)

schema_gen = SchemaGen(
    statistics=statistics_gen.outputs["statistics"],
    infer_feature_shape=True,
)
context.run(schema_gen)

example_validator = ExampleValidator(
    statistics=statistics_gen.outputs["statistics"],
    schema=schema_gen.outputs["schema"],
)
context.run(example_validator)


ExecutionResult(
    component_id: ExampleValidator
    execution_id: 14
    outputs:
        anomalies: OutputChannel(artifact_type=ExampleAnomalies, producer_component_id=ExampleValidator, output_key=anomalies, additional_properties={}, additional_custom_properties={}, _input_trigger=None)

## 5. Transform

Komponen `Transform` menjalankan preprocessing pada fitur numerik. Pada proyek ini, setiap fitur numerik distandardisasi menggunakan z-score melalui `tft.scale_to_z_score`.


In [15]:
transform = Transform(
    examples=example_gen.outputs["examples"],
    schema=schema_gen.outputs["schema"],
    module_file=TRANSFORM_MODULE,
)
context.run(transform)


INFO:tensorflow:Assets written to: D:\SERTIFIKAT DICODING\Mlops\Proyek pengembangan machine learning pipeline\fahrual_19-pipeline\Transform\transform_graph\15\.temp_path\tftransform_tmp\fba3528c04854b16b87d9d544395e91a\assets


INFO:tensorflow:Assets written to: D:\SERTIFIKAT DICODING\Mlops\Proyek pengembangan machine learning pipeline\fahrual_19-pipeline\Transform\transform_graph\15\.temp_path\tftransform_tmp\fba3528c04854b16b87d9d544395e91a\assets


INFO:tensorflow:struct2tensor is not available.


INFO:tensorflow:struct2tensor is not available.


INFO:tensorflow:tensorflow_decision_forests is not available.


...

INFO:tensorflow:tensorflow_decision_forests is not available.


INFO:tensorflow:tensorflow_decision_forests is not available.


INFO:tensorflow:tensorflow_text is not available.


INFO:tensorflow:tensorflow_text is not available.


ExecutionResult(
    component_id: Transform
    execution_id: 15
    outputs:
        transform_graph: OutputChannel(artifact_type=TransformGraph, producer_component_id=Transform, output_key=transform_graph, additional_properties={}, additional_custom_properties={}, _input_trigger=None
        transformed_examples: OutputChannel(artifact_type=Examples, producer_component_id=Transform, output_key=transformed_examples, additional_properties={}, additional_custom_properties={}, _input_trigger=None
        updated_analyzer_cache: OutputChannel(artifact_type=TransformCache, producer_component_id=Transform, output_key=updated_analyzer_cache, additional_properties={}, additional_custom_properties={}, _input_trigger=None
        pre_transform_schema: OutputChannel(artifact_type=Schema, producer_component_id=Transform, output_key=pre_transform_schema, additional_properties={}, additional_custom_properties={}, _input_trigger=None
        pre_transform_stats: OutputChannel(artifact_type=ExampleStatistics, producer_component_id=Transform, output_key=pre_transform_stats, additional_properties={}, additional_custom_properties={}, _input_trigger=None
        post_transform_schema: OutputChannel(artifact_type=Schema, producer_component_id=Transform, output_key=post_transform_schema, additional_properties={}, additional_custom_properties={}, _input_trigger=None
        post_transform_stats: OutputChannel(artifact_type=ExampleStatistics, producer_component_id=Transform, output_key=post_transform_stats, additional_properties={}, additional_custom_properties={}, _input_trigger=None
        post_transform_anomalies: OutputChannel(artifact_type=ExampleAnomalies, producer_component_id=Transform, output_key=post_transform_anomalies, additional_properties={}, additional_custom_properties={}, _input_trigger=None)

## 6. Trainer

Komponen `Trainer` melatih model neural network sederhana menggunakan TensorFlow/Keras. Model menerima empat fitur numerik hasil transformasi dan menghasilkan prediksi tiga kelas spesies Iris.


In [16]:
trainer = Trainer(
    module_file=TRAINER_MODULE,
    examples=transform.outputs["transformed_examples"],
    transform_graph=transform.outputs["transform_graph"],
    schema=schema_gen.outputs["schema"],
    train_args=trainer_pb2.TrainArgs(num_steps=20),
    eval_args=trainer_pb2.EvalArgs(num_steps=5),
    custom_config={"num_epochs": 30},
)
context.run(trainer)


 1/20 [>.............................] - ETA: 10s - loss: 1.1504 - accuracy: 0.2500WARNING:tensorflow:Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches (in this case, 5 batches). You may need to use the repeat() function when building your dataset.


20/20 [==============================] - 1s 13ms/step - loss: 0.6705 - accuracy: 0.7328 - val_loss: 0.4116 - val_accuracy: 0.8500
INFO:tensorflow:struct2tensor is not available.


INFO:tensorflow:struct2tensor is not available.


INFO:tensorflow:tensorflow_decision_forests is not available.


INFO:tensorflow:tensorflow_decision_forests is not available.


INFO:tensorflow:tensorflow_text is not available.


INFO:tensorflow:tensorflow_text is not available.


INFO:tensorflow:Assets written to: D:\SERTIFIKAT DICODING\Mlops\Proyek pengembangan machine learning pipeline\fahrual_19-pipeline\Trainer\model\16\Format-Serving\assets


INFO:tensorflow:Assets written to: D:\SERTIFIKAT DICODING\Mlops\Proyek pengembangan machine learning pipeline\fahrual_19-pipeline\Trainer\model\16\Format-Serving\assets


ExecutionResult(
    component_id: Trainer
    execution_id: 16
    outputs:
        model: OutputChannel(artifact_type=Model, producer_component_id=Trainer, output_key=model, additional_properties={}, additional_custom_properties={}, _input_trigger=None
        model_run: OutputChannel(artifact_type=ModelRun, producer_component_id=Trainer, output_key=model_run, additional_properties={}, additional_custom_properties={}, _input_trigger=None)

## 7. Resolver dan Evaluator

`Resolver` mencari model terbaik sebelumnya sebagai baseline. `Evaluator` mengevaluasi model baru menggunakan TensorFlow Model Analysis.


In [17]:
model_resolver = Resolver(
    strategy_class=latest_blessed_model_resolver.LatestBlessedModelResolver,
    model=Channel(type=Model),
    model_blessing=Channel(type=ModelBlessing),
).with_id("latest_blessed_model_resolver")
context.run(model_resolver)

eval_config = tfma.EvalConfig(
    model_specs=[tfma.ModelSpec(label_key="species")],
    slicing_specs=[tfma.SlicingSpec()],
    metrics_specs=[
        tfma.MetricsSpec(
            metrics=[
                tfma.MetricConfig(
                    class_name="SparseCategoricalAccuracy",
                    threshold=tfma.MetricThreshold(
                        value_threshold=tfma.GenericValueThreshold(
                            lower_bound={"value": 0.5}
                        )
                    ),
                ),
                tfma.MetricConfig(class_name="ExampleCount"),
            ]
        )
    ],
)

evaluator = Evaluator(
    examples=example_gen.outputs["examples"],
    model=trainer.outputs["model"],
    baseline_model=model_resolver.outputs["model"],
    eval_config=eval_config,
)
context.run(evaluator, enable_cache=False)


Instructions for updating:
Use eager execution and: 
`tf.data.TFRecordDataset(path)`


Instructions for updating:
Use eager execution and: 
`tf.data.TFRecordDataset(path)`


ExecutionResult(
    component_id: Evaluator
    execution_id: 18
    outputs:
        evaluation: OutputChannel(artifact_type=ModelEvaluation, producer_component_id=Evaluator, output_key=evaluation, additional_properties={}, additional_custom_properties={}, _input_trigger=None
        blessing: OutputChannel(artifact_type=ModelBlessing, producer_component_id=Evaluator, output_key=blessing, additional_properties={}, additional_custom_properties={}, _input_trigger=None)

## 8. Pusher

Komponen `Pusher` menyimpan model yang sudah lolos evaluasi ke folder `serving_model`. Folder ini dapat digunakan sebagai model serving.


In [18]:
pusher = Pusher(
    model=trainer.outputs["model"],
    model_blessing=evaluator.outputs["blessing"],
    push_destination=pusher_pb2.PushDestination(
        filesystem=pusher_pb2.PushDestination.Filesystem(
            base_directory=SERVING_MODEL_DIR
        )
    ),
)
context.run(pusher, enable_cache=False)


ExecutionResult(
    component_id: Pusher
    execution_id: 19
    outputs:
        pushed_model: OutputChannel(artifact_type=PushedModel, producer_component_id=Pusher, output_key=pushed_model, additional_properties={}, additional_custom_properties={}, _input_trigger=None)

## 9. Ringkasan

Notebook ini sudah memuat komponen wajib `ExampleGen`, `StatisticsGen`, `SchemaGen`, `ExampleValidator`, `Transform`, `Trainer`, `Resolver`, `Evaluator`, dan `Pusher`. Setelah semua cell dijalankan, artifact pipeline tersimpan di `fahrual_19-pipeline` dan model siap serving tersimpan di `serving_model`.
